In [1]:
# %%
# =============================================================================
# kSZ²-21cm : ION_Tvir_MIN SCAN (QMA Style)
# Fixed HII_EFF_FACTOR=10.0, varying ION_Tvir_MIN
# Extended to z=0.0001 for full kSZ integration
# Parallel over ION_Tvir_MIN + Multi-seed support
# =============================================================================

import numpy as np
import matplotlib as mpl
import matplotlib
import matplotlib.pyplot as plt

import py21cmfast as p21c
import os
import glob
import time
from datetime import datetime

# PBS vs desktop backend
if os.environ.get('PBS_JOBID'):
    matplotlib.use('Agg')
    print("✓ Using Agg backend (PBS/server mode)")
else:
    matplotlib.use('Agg')
    print("✓ Using Agg backend")

print(f"py21cmfast version: {p21c.__version__}")

# =============================================================================
# CELL 1a: Output and Cache Directories (QMA Style)
# =============================================================================

plot_dir = "11May2026_kSZ2_21cm_HII_EFF_scan/plots"
if not os.path.exists(plot_dir):
    os.makedirs(plot_dir)
    print(f"Created directory: {plot_dir}")
else:
    print(f"Directory already exists: {plot_dir}")
print(f"All plots will be saved to: {os.path.abspath(plot_dir)}")

# --- Cache directory (PBS-aware + Notebook aware) ---
try:
    # Running as .py script via PBS
    main_cache_dir = os.path.join(
        os.path.dirname(os.path.abspath(__file__)), 
        "11May2026_kSZ2_21cm_HII_EFF_scan", "cache"
    )
except NameError:
    # Running in Jupyter notebook
    main_cache_dir = os.path.join(
        os.getcwd(), 
        "11May2026_kSZ2_21cm_HII_EFF_scan", "cache"
    )
os.makedirs(main_cache_dir, exist_ok=True)
print(f"Cache directory: {main_cache_dir}")
# =============================================================================
# CELL 1b: Global Plot Settings (QMA Style - DO NOT override later)
# =============================================================================

plt.rcParams.update({
    # Font
    'font.family'        : 'serif',
    'font.serif'         : ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset'   : 'cm',
    'font.size'          : 20,
    'axes.labelsize'     : 28,
    'axes.titlesize'     : 22,
    'xtick.labelsize'    : 22,
    'ytick.labelsize'    : 22,
    'legend.fontsize'    : 18,
    'figure.titlesize'   : 20,
    # Ticks
    'xtick.direction'    : 'in',
    'ytick.direction'    : 'in',
    'xtick.major.size'   : 6,
    'ytick.major.size'   : 6,
    'xtick.minor.size'   : 3,
    'ytick.minor.size'   : 3,
    'xtick.major.width'  : 1.0,
    'ytick.major.width'  : 1.0,
    'xtick.minor.width'  : 0.8,
    'ytick.minor.width'  : 0.8,
    'xtick.top'          : True,
    'ytick.right'        : True,
    'xtick.minor.visible': True,
    'ytick.minor.visible': True,
    # Lines / axes
    'axes.linewidth'     : 1.0,
    'lines.linewidth'    : 1.8,
    'lines.markersize'   : 5,
    # Grid — OFF everywhere
    'axes.grid'          : False,
    # Figure / save
    'figure.dpi'         : 150,
    'savefig.dpi'        : 300,
    'savefig.bbox'       : 'tight',
    'savefig.pad_inches' : 0.05,
})

print("✓ Global plot settings applied (grid OFF, no downstream overrides needed)")

# =============================================================================
# PDF / PNG style contexts + save_pdf_png
# =============================================================================

PDF_STYLE = {
    'font.family'        : 'serif',
    'font.serif'         : ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset'   : 'cm',
    'font.size'          : 28,
    'axes.labelsize'     : 28,
    'axes.titlesize'     : 32,
    'xtick.labelsize'    : 26,
    'ytick.labelsize'    : 26,
    'legend.fontsize'    : 22,
    'figure.titlesize'   : 28,
    'xtick.direction'    : 'in',
    'ytick.direction'    : 'in',
    'xtick.top'          : True,
    'ytick.right'        : True,
    'xtick.minor.visible': True,
    'ytick.minor.visible': True,
    'xtick.major.size'   : 6,
    'ytick.major.size'   : 6,
    'xtick.minor.size'   : 3,
    'ytick.minor.size'   : 3,
    'axes.linewidth'     : 1.0,
    'lines.linewidth'    : 1.8,
    'axes.grid'          : False,
    'figure.dpi'         : 150,
    'savefig.dpi'        : 300,
    'savefig.bbox'       : 'tight',
    'savefig.pad_inches' : 0.05,
}

PNG_STYLE = {
    'font.family'        : 'serif',
    'font.serif'         : ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset'   : 'cm',
    'font.size'          : 16,
    'axes.labelsize'     : 22,
    'axes.titlesize'     : 18,
    'xtick.labelsize'    : 20,
    'ytick.labelsize'    : 20,
    'legend.fontsize'    : 18,
    'figure.titlesize'   : 16,
    'xtick.direction'    : 'in',
    'ytick.direction'    : 'in',
    'xtick.top'          : True,
    'ytick.right'        : True,
    'xtick.minor.visible': True,
    'ytick.minor.visible': True,
    'axes.linewidth'     : 1.0,
    'lines.linewidth'    : 1.5,
    'axes.grid'          : False,
    'figure.dpi'         : 150,
    'savefig.dpi'        : 300,
    'savefig.bbox'       : 'tight',
    'savefig.pad_inches' : 0.05,
}

def save_pdf_png(plot_func, plot_dir, plot_name, title=None, figsize=(10, 7)):
    """
    Save a plot as both PDF and PNG using QMA style.
    
    Parameters
    ----------
    plot_func : callable
        f(ax) — draws onto the provided Axes.
        Do NOT set font sizes or grid inside plot_func.
    plot_dir  : str
        Directory to save files
    plot_name : str
        Filename without extension
    title     : str or None
        PNG-only title (PDF has no title)
    figsize   : tuple, optional
        Figure size (width, height) in inches. Default: (10, 7)
    """
    with mpl.rc_context(PDF_STYLE):
        fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
        plot_func(ax)
        ax.set_title("")
        ax.grid(False)
        fig.savefig(f"{plot_dir}/{plot_name}.pdf")
        plt.close(fig)

    with mpl.rc_context(PNG_STYLE):
        fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
        plot_func(ax)
        ax.grid(False)
        if title is not None:
            ax.set_title(title, fontweight='bold')
        fig.savefig(f"{plot_dir}/{plot_name}.png")
        plt.close(fig)

print("✓ save_pdf_png defined (plot_func pattern, grid always OFF, figsize customizable)")

# =============================================================================
# CELL 1c: Define Parameters
# =============================================================================

# Test mode for development
TEST_MODE = False

if TEST_MODE:
    print("\n" + "="*70)
    print("🔧 TEST MODE - Small box")
    print("="*70)
    box_len = 100.0
    hii_dim = 32
    n_threads = 8
else:
    print("\n" + "="*70)
    print("🚀 PRODUCTION MODE - Full resolution")
    print("="*70)
    box_len = 800.0
    hii_dim = 128
    n_threads = 64

user_params = p21c.UserParams(
    HII_DIM=hii_dim,
    BOX_LEN=box_len,
    USE_INTERPOLATION_TABLES=True,
    N_THREADS=n_threads
)

z_min = 5
z_max = 20.0

# Multi-seed setup
RANDOM_SEEDS = list(range(1, 11))
N_SEEDS = len(RANDOM_SEEDS)

# HII_EFF_FACTOR scan
HII_EFF_FACTOR_VALUES = np.linspace(25.0, 70.0, 11)

if TEST_MODE:
    HII_EFF_FACTOR_VALUES = HII_EFF_FACTOR_VALUES[::2]

print(f"\n=== PARAMETER SCAN SETUP (kSZ²-21cm) ===")
#print(f"Fixed HII_EFF_FACTOR = {HII_EFF_FACTOR_FIXED}")
#print(f"Scanning {len(ION_Tvir_MIN_VALUES)} values of ION_Tvir_MIN")
print(f"Multi-seed: {N_SEEDS} realisations per ION_Tvir_MIN")
#print(f"Total simulations: {len(ION_Tvir_MIN_VALUES) * N_SEEDS}")
print(f"Box: {box_len} Mpc, Resolution: {hii_dim}³")
print(f"z range: {z_min} → {z_max}")

print("\n=== MULTI-SEED SETUP ===")
print(f"Seeds: {RANDOM_SEEDS}")
print(f"Total realisations: {N_SEEDS}")

print("\n=== DEFAULT COSMOLOGY ===")
print(p21c.CosmoParams())

print("\n=== DEFAULT ASTROPHYSICS ===")
print(p21c.AstroParams())

print("\n" + "="*70)

✓ Using Agg backend (PBS/server mode)
py21cmfast version: 3.3.1
Directory already exists: 11May2026_kSZ2_21cm_HII_EFF_scan/plots
All plots will be saved to: /user1/swanith/11May2026_kSZ2_21cm_HII_EFF_scan/plots
Cache directory: /user1/swanith/11May2026_kSZ2_21cm_HII_EFF_scan/cache
✓ Global plot settings applied (grid OFF, no downstream overrides needed)
✓ save_pdf_png defined (plot_func pattern, grid always OFF, figsize customizable)

🚀 PRODUCTION MODE - Full resolution

=== PARAMETER SCAN SETUP (kSZ²-21cm) ===
Multi-seed: 10 realisations per ION_Tvir_MIN
Box: 800.0 Mpc, Resolution: 128³
z range: 5 → 20.0

=== MULTI-SEED SETUP ===
Seeds: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Total realisations: 10

=== DEFAULT COSMOLOGY ===
CosmoParams:
    OMb        : 0.04897468161869667
    OMm        : 0.30964144154550644
    POWER_INDEX: 0.9665
    SIGMA_8    : 0.8102
    hlittle    : 0.6766
    

=== DEFAULT ASTROPHYSICS ===
AstroParams:
    ALPHA_ESC       : -0.5
    ALPHA_STAR      : 0.5
    ALPHA_ST

/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/py21cmfast/_cfg.py:57: UserWarning: Your configuration file is out of date. Updating...
  warnings.warn(
/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/py21cmfast/_cfg.py:41: UserWarning: Your configuration file is out of date. Updating...
  warnings.warn("Your configuration file is out of date. Updating...")


In [4]:
# %%
# =============================================================================
# CELL 2: Parallel Lightcone Generation — Multi-Seed + HII_EFF_FACTOR Scan
# QMA Style: Robust caching, fork context, progress tracking
# FIXED: Returns cache file paths instead of CFFI objects (unpicklable)
# =============================================================================

import time
import glob
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp

print("\n" + "="*70)
print("RUNNING PARALLEL LIGHTCONE SIMULATIONS (Seeds × HII_EFF_FACTOR)")
print("="*70)

# =============================================================================
# Updated Directories (11May2026 as requested)
# =============================================================================
plot_dir = "11May2026_kSZ2_21cm_HII_EFF_scan/plots"
main_cache_dir = "11May2026_kSZ2_21cm_HII_EFF_scan/cache"

os.makedirs(plot_dir, exist_ok=True)
os.makedirs(main_cache_dir, exist_ok=True)

print(f"Plots → {os.path.abspath(plot_dir)}")
print(f"Cache → {os.path.abspath(main_cache_dir)}")

# =============================================================================
# Parallel Setup
# =============================================================================
N_TOTAL_CORES = int(os.environ.get('PBS_NCPUS', os.cpu_count() or 16))
N_THREADS_PER_WORKER = 8
N_WORKERS = max(1, N_TOTAL_CORES // N_THREADS_PER_WORKER)
N_WORKERS = min(N_WORKERS, len(HII_EFF_FACTOR_VALUES) * N_SEEDS)

# Allow override (e.g., N_WORKERS = 2 for memory constraints)
# N_WORKERS = 2

print(f"Parallel Execution: {N_WORKERS} workers × {N_THREADS_PER_WORKER} threads")
print(f"CPU cores available: {N_TOTAL_CORES}")

# =============================================================================
# Worker Function (Top-level for pickling)
# FIXED: Returns cache file path instead of lightcone object
# =============================================================================
def _run_or_load_lightcone(seed, hii_eff, cache_base_dir, z_min, z_max, user_params, n_threads_worker):
    """
    Run or load one (seed, HII_EFF_FACTOR) combination.
    
    Returns cache file path (not lightcone object, which can't be pickled)
    
    Parameters
    ----------
    seed : int
        Random seed
    hii_eff : float
        HII_EFF_FACTOR value
    cache_base_dir : str
        Base cache directory (main_cache_dir)
    z_min, z_max : float
        Redshift range
    user_params : py21cmfast.UserParams
        User parameters
    ion_tvir_min_fixed : float
        Fixed ION_Tvir_MIN value
    n_threads_worker : int
        Number of threads for this worker
    
    Returns
    -------
    tuple : (seed, hii_eff, cache_file_path or None, status_str, sim_time_seconds)
    """
    import os
    import glob
    import time as _time
    import py21cmfast as _p21c

    # Create unique subdirectory: seed_XXX_HIIEffX.XXX
    cache_subdir = os.path.join(
        cache_base_dir, 
        f"seed_{seed:03d}_HIIEff{hii_eff:.4f}"
    )
    os.makedirs(cache_subdir, exist_ok=True)

    # Build astrophysical parameters for this HII_EFF_FACTOR value
    astro_params = _p21c.AstroParams(
        HII_EFF_FACTOR=hii_eff
        
    )

    # ==========================================================================
    # CACHE CHECK (native py21cmfast HDF5)
    # ==========================================================================
    cached_files = sorted(glob.glob(os.path.join(cache_subdir, "LightCone_*.h5")))
    valid_cached = [(f, os.path.getsize(f) / 1e6)
                    for f in cached_files if os.path.getsize(f) / 1e6 > 1.0]

    if valid_cached:
        cache_file, size_mb = valid_cached[0]
        # Validate it's loadable by re-running with write=False (uses cache)
        try:
            lc = _p21c.run_lightcone(
                redshift=z_min,
                max_redshift=z_max,
                lightcone_quantities=('brightness_temp', 'density', 'xH_box', 'velocity'),
                user_params=user_params,
                astro_params=astro_params,
                random_seed=seed,
                direc=cache_subdir,
                write=False,
            )
            # Return cache file path, not lightcone object (CFFI can't pickle)
            return (seed, hii_eff, cache_file, "cached", 0.0)
        except Exception as e:
            # Cache file present but invalid — fall through to recompute
            pass

    # ==========================================================================
    # RUN NEW SIMULATION
    # ==========================================================================
    sim_start = _time.time()
    try:
        lc = _p21c.run_lightcone(
            redshift=z_min,
            max_redshift=z_max,
            lightcone_quantities=('brightness_temp', 'density', 'xH_box', 'velocity'),
            user_params=user_params,
            astro_params=astro_params,
            random_seed=seed,
            direc=cache_subdir,
            write=True
        )
        
        # Make sure it's persisted
        try:
            lc.save(direc=cache_subdir)
        except Exception:
            pass

        # Get the cache file path
        cached_files = sorted(glob.glob(os.path.join(cache_subdir, "LightCone_*.h5")))
        cache_file = cached_files[0] if cached_files else None

        sim_time = _time.time() - sim_start
        # Return cache file path, not lightcone object (CFFI can't pickle)
        return (seed, hii_eff, cache_file, "computed", sim_time)
    
    except Exception as e:
        sim_time = _time.time() - sim_start
        return (seed, hii_eff, None, f"failed: {str(e)}", sim_time)


# =============================================================================
# Dispatch Parallel Jobs
# =============================================================================
lightcones = {}   # key: (seed, hii_eff) → value: lightcone object (loaded in main process)

scan_start = time.time()
mp_ctx = mp.get_context("fork")

total_jobs = len(RANDOM_SEEDS) * len(HII_EFF_FACTOR_VALUES)
print(f"\nDispatching {total_jobs} jobs:")
print(f"  Seeds: {len(RANDOM_SEEDS)} ({min(RANDOM_SEEDS)}–{max(RANDOM_SEEDS)})")
print(f"  HII_EFF_FACTOR values: {len(HII_EFF_FACTOR_VALUES)}")
print(f"  Workers: {N_WORKERS} (fork context)")
print()

with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=mp_ctx) as ex:
    futures = {}
    for seed in RANDOM_SEEDS:
        for hii_eff in HII_EFF_FACTOR_VALUES:
            # Submit job to worker pool
            fut = ex.submit(
                _run_or_load_lightcone,
                seed,
                hii_eff,
                main_cache_dir,
                z_min,
                z_max,
                user_params,
                N_THREADS_PER_WORKER  # This was already there
            )
            futures[fut] = (seed, hii_eff)

    # Track results as they complete
    completed = 0
    cached_count = 0
    computed_count = 0
    failed_count = 0

    for fut in as_completed(futures):
        seed, hii_eff, cache_file, status, sim_time = fut.result()
        key = (seed, hii_eff)
        
        # Load lightcone from cache file in MAIN PROCESS (not in worker)
        # This avoids pickling CFFI objects
        if cache_file and (status == "cached" or status == "computed"):
            try:
                lightcones[key] = p21c.LightCone.read(cache_file)
            except Exception as e:
                lightcones[key] = None
                status = f"failed to load: {e}"
        else:
            lightcones[key] = None

        completed += 1
        
        # Format status message
        if status == "cached":
            msg = "✓ cached"
            cached_count += 1
        elif status == "computed":
            msg = f"✓ computed ({sim_time/60:.2f} min)"
            computed_count += 1
        else:
            msg = f"✗ {status}"
            failed_count += 1

        # Progress line
        elapsed = (time.time() - scan_start) / 60
        pct = 100.0 * completed / len(futures)
        print(f"  [{completed:3d}/{len(futures):3d} ({pct:5.1f}%)] "
              f"seed={seed:2d}, HIIEff={hii_eff:.4f} → {msg} "
              f"(elapsed: {elapsed:6.1f} min)")

total_time = time.time() - scan_start
print(f"\n{'='*70}")
print(f"✓ ALL LIGHTCONES COMPLETE")
print(f"{'='*70}")
print(f"  Total time:  {total_time/60:.2f} minutes")
print(f"  Cached:      {cached_count}")
print(f"  Computed:    {computed_count}")
print(f"  Failed:      {failed_count}")
print(f"  Successful:  {sum(1 for v in lightcones.values() if v is not None)}/{len(lightcones)}")
print(f"{'='*70}")

# Store cache_dir for later cells
cache_dir = main_cache_dir
print(f"\n✓ Cache directory for later cells: {cache_dir}")
print(f"✓ Lightcones dict ready: {len(lightcones)} entries")


RUNNING PARALLEL LIGHTCONE SIMULATIONS (Seeds × HII_EFF_FACTOR)
Plots → /user1/swanith/11May2026_kSZ2_21cm_HII_EFF_scan/plots
Cache → /user1/swanith/11May2026_kSZ2_21cm_HII_EFF_scan/cache
Parallel Execution: 8 workers × 8 threads
CPU cores available: 64

Dispatching 110 jobs:
  Seeds: 10 (1–10)
  HII_EFF_FACTOR values: 11
  Workers: 8 (fork context)



/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/py21cmfast/_utils.py:400: UserWarning: The following parameters to FlagOptions are not supported: ['USE_VELS_AUX']
  warnings.warn(
/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/py21cmfast/_utils.py:400: UserWarning: The following parameters to FlagOptions are not supported: ['USE_VELS_AUX']
  warnings.warn(
/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/py21cmfast/_utils.py:400: UserWarning: The following parameters to FlagOptions are not supported: ['USE_VELS_AUX']
  warnings.warn(
/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/py21cmfast/_utils.py:400: UserWarning: The following parameters to FlagOptions are not supported: ['USE_VELS_AUX']
  warnings.warn(
/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/py21cmfast/_utils.py:400: UserWarning: The following parameters to FlagOptions are not supported: ['USE_VELS_AUX']
  warnings.warn(
/user1/swanith/.cond

  [  1/110 (  0.9%)] seed= 1, HIIEff=52.0000 → ✓ computed (5.85 min) (elapsed:    5.9 min)
  [  2/110 (  1.8%)] seed= 1, HIIEff=34.0000 → ✓ computed (5.88 min) (elapsed:    5.9 min)
  [  3/110 (  2.7%)] seed= 1, HIIEff=38.5000 → ✓ computed (5.90 min) (elapsed:    5.9 min)
  [  4/110 (  3.6%)] seed= 1, HIIEff=56.5000 → ✓ computed (5.92 min) (elapsed:    5.9 min)
  [  5/110 (  4.5%)] seed= 1, HIIEff=43.0000 → ✓ computed (5.92 min) (elapsed:    5.9 min)
  [  6/110 (  5.5%)] seed= 1, HIIEff=29.5000 → ✓ computed (5.97 min) (elapsed:    6.0 min)
  [  7/110 (  6.4%)] seed= 1, HIIEff=47.5000 → ✓ cached (elapsed:    7.4 min)
  [  8/110 (  7.3%)] seed= 1, HIIEff=25.0000 → ✓ cached (elapsed:    7.6 min)
  [  9/110 (  8.2%)] seed= 2, HIIEff=29.5000 → ✓ computed (7.81 min) (elapsed:   13.7 min)
  [ 10/110 (  9.1%)] seed= 1, HIIEff=61.0000 → ✓ computed (7.97 min) (elapsed:   13.8 min)
  [ 11/110 ( 10.0%)] seed= 1, HIIEff=65.5000 → ✓ computed (8.07 min) (elapsed:   14.0 min)
  [ 12/110 ( 10.9%)] seed

In [6]:
# %%
# =============================================================================
# CELL 4: Reionization History + Optical Depth (Per Seed)
# QMA Style - No early averaging over seeds
# FIXED: Store ds_Mpc in tau_results for use in Cell 6
# =============================================================================

print("\n" + "="*70)
print("REIONIZATION HISTORY + OPTICAL DEPTH ANALYSIS (Per Seed)")
print("="*70)

# =============================================================================
# Optical Depth Prefactor (Thomson cross-section)
# =============================================================================
prefactor = 3.0e-7

# =============================================================================
# Compute per (seed, hii_eff) 
# =============================================================================
tau_results = {}  # key: (seed, hii_eff)

for (seed, hii_eff), lc in lightcones.items():
    if lc is None:
        continue
    
    print(f"  Processing seed={seed}, HII_EFF_FACTOR={hii_eff:.3f}", end="\r")
    
    # Geometry
    red_axis = np.asarray(lc.lightcone_redshifts)
    pos_axis = np.asarray(lc.lightcone_distances)
    ind_z = np.where(red_axis <= z_max)[0]
    red_axis = red_axis[ind_z]
    pos_axis = pos_axis[ind_z]
    
    # Ionization history
    z_nodes = np.asarray(lc.node_redshifts[::-1])
    x_e_nodes = 1.0 - np.asarray(lc.global_xH[::-1])
    
    # Interpolate
    x_e_interp = np.interp(red_axis, z_nodes, x_e_nodes)
    
    # Integration
    ds_Mpc = np.diff(pos_axis)
    z_mid = 0.5 * (red_axis[:-1] + red_axis[1:])
    x_e_mid = 0.5 * (x_e_interp[:-1] + x_e_interp[1:])
    dtau = prefactor * x_e_mid * (1.0 + z_mid)**2 * ds_Mpc
    tau = np.cumsum(dtau)
    
    tau_results[(seed, hii_eff)] = {
        'z_mid': z_mid,
        'tau': tau,
        'tau_total': tau[-1],
        'red_axis': red_axis,
        'z_nodes': z_nodes,
        'x_e_nodes': x_e_nodes,
        'ds_Mpc': ds_Mpc
    }

print(f"\n✓ Computed optical depth for {len(tau_results)} realisations "
      f"({N_SEEDS} seeds × {len(HII_EFF_FACTOR_VALUES)} HII_EFF_FACTOR)")

# =============================================================================
# PLOT 4a: x_e vs z (one line per (seed, hii_eff))
# =============================================================================

def _draw_xe(ax):
    cmap = mpl.cm.plasma
    norm = mpl.colors.Normalize(vmin=min(HII_EFF_FACTOR_VALUES), 
                                vmax=max(HII_EFF_FACTOR_VALUES))
    
    for (seed, hii_eff), data in tau_results.items():
        color = cmap(norm(hii_eff))
        ax.plot(data['z_nodes'], data['x_e_nodes'], 
                color=color, lw=1.2, alpha=0.6)
    
    ax.set_xlabel(r'Redshift $z$')
    ax.set_ylabel(r'Ionization Fraction $x_e$')
    ax.set_ylim(-0.05, 1.05)
    ax.invert_xaxis()
    
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = ax.figure.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label(r'HII\_EFF\_FACTOR')

save_pdf_png(
    _draw_xe, plot_dir,
    "reionization_history_xe_HII_EFF_all",
    title=f'Reionization History: Ionization Fraction )'
)
print("✓ Saved: reionization_history_xe_HII_EFF_all")

# =============================================================================
# PLOT 4b: Cumulative τ vs z
# =============================================================================

def _draw_tau(ax):
    cmap = mpl.cm.plasma
    norm = mpl.colors.Normalize(vmin=min(HII_EFF_FACTOR_VALUES), 
                                vmax=max(HII_EFF_FACTOR_VALUES))
    
    for (seed, hii_eff), data in tau_results.items():
        color = cmap(norm(hii_eff))
        ax.plot(data['z_mid'], data['tau'], color=color, lw=1.2, alpha=0.6)
    
    ax.set_xlabel(r'Redshift $z$')
    ax.set_ylabel(r'Cumulative Optical Depth $\tau(<z)$')
    ax.invert_xaxis()
    
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = ax.figure.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label(r'HII\_EFF\_FACTOR')

save_pdf_png(
    _draw_tau, plot_dir,
    "tau_vs_z_HII_EFF_all",
    title=f'Cumulative Optical Depth vs Redshift )'
)
print("✓ Saved: tau_vs_z_HII_EFF_all")

print("\n" + "="*70)
print("✓ CELL 4 COMPLETE")
print("="*70)


REIONIZATION HISTORY + OPTICAL DEPTH ANALYSIS (Per Seed)
  Processing seed=10, HII_EFF_FACTOR=70.000
✓ Computed optical depth for 110 realisations (10 seeds × 11 HII_EFF_FACTOR)
✓ Saved: reionization_history_xe_HII_EFF_all
✓ Saved: tau_vs_z_HII_EFF_all

✓ CELL 4 COMPLETE


In [7]:
# %%
# =============================================================================
# CELL 5: Compute kSZ Integrand with Visibility Function (Per Seed)
# =============================================================================

print("\n" + "="*70)
print("COMPUTING kSZ INTEGRAND WITH VISIBILITY FUNCTION (Per Seed)")
print("="*70)

c_Mpc_s = 299792.458 / 3.08567758e19
print(f"Speed of light: c = {c_Mpc_s:.6e} Mpc/s")

kSZ_integrand_results = {}   # key: (seed, hii_eff)

for (seed, hii_eff), lc in lightcones.items():
    if lc is None:
        continue
    
    if (seed, hii_eff) not in tau_results:
        continue
    
    tau_data = tau_results[(seed, hii_eff)]
    
    # Extract fields
    red_axis_full = np.asarray(lc.lightcone_redshifts, dtype=np.float64)
    ind_z = np.where(red_axis_full <= z_max)[0]
    density_1plus = 1 + np.asarray(lc.density[:, :, ind_z])
    x_e_3D        = 1 - np.asarray(lc.xH_box[:, :, ind_z])
    v_los_Mpc_s   = np.asarray(lc.velocity[:, :, ind_z]) / 67.4
    
    # Tau interpolation
    red_axis = np.asarray(tau_data['red_axis'], dtype=np.float64)
    z_mid    = np.asarray(tau_data['z_mid'], dtype=np.float64)
    tau      = np.asarray(tau_data['tau'], dtype=np.float64)
    tau_extended = np.concatenate([[0.0], tau])
    z_extended   = np.concatenate([[red_axis[0]], z_mid])
    tau_at_lc    = np.interp(red_axis, z_extended, tau_extended)
    visibility = np.exp(-tau_at_lc)
    visibility_3D = visibility[None, None, :]
    
    # Compute integrand
    kSZ_integrand = (density_1plus * x_e_3D * 
                     v_los_Mpc_s / c_Mpc_s * 
                     visibility_3D)
    
    kSZ_integrand_results[(seed, hii_eff)] = {
        'kSZ_integrand': kSZ_integrand,
        'visibility': visibility,
        'red_axis': red_axis,
        'ind_z': ind_z
    }

print(f"✓ Computed kSZ integrand for {len(kSZ_integrand_results)} realisations")

# =============================================================================
# PLOT 5a: kSZ Integrand (Representative)
# =============================================================================

print("\nGenerating kSZ Integrand Plot...")
hii_eff_subset = HII_EFF_FACTOR_VALUES[::max(1, len(HII_EFF_FACTOR_VALUES)//4)]

def _draw_integrand(ax):
    # Pick one representative (seed=1) for each hii_eff in subset
    for hii_eff in hii_eff_subset:
        found = False
        for s in RANDOM_SEEDS:
            key = (s, hii_eff)
            if key in kSZ_integrand_results:
                data = kSZ_integrand_results[key]
                found = True
                break
        
        if not found:
            continue
        
        kSZ_integrand = data['kSZ_integrand']
        ind_z = data['ind_z']
        lc = lightcones[(s, hii_eff)]
        slice_2D = kSZ_integrand[:, :, kSZ_integrand.shape[2]//2]
        x_extent = float(np.asarray(lc.lightcone_distances[ind_z].max()))
        y_extent = float(user_params.BOX_LEN)
        vmax = float(np.percentile(np.abs(kSZ_integrand), 99))
        
        im = ax.imshow(slice_2D.T, extent=[0, x_extent, 0, y_extent],
                       aspect='auto', cmap='seismic', origin='lower',
                       vmin=-vmax, vmax=vmax)
        ax.figure.colorbar(im, ax=ax, fraction=0.046).set_label(
            r'kSZ Integrand')
        ax.set_xlabel('Comoving Distance [Mpc]')
        ax.set_ylabel('Comoving Distance [Mpc]')
        ax.text(0.02, 0.98, f'HII_EFF_FACTOR={hii_eff:.2f} (seed={s})',
                transform=ax.transAxes, fontsize=13, fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        break  # plot only one per hii_eff

save_pdf_png(
    _draw_integrand, plot_dir,
    "kSZ_integrand_with_visibility_HII_EFF",
    title=r'kSZ Integrand $(1+\delta) x_e v_z/c \, e^{-\tau}$',
    figsize=(13, 8)
)
print("✓ Saved: kSZ_integrand_with_visibility_HII_EFF")
print(f"  DEBUG: kSZ_integrand shape = {kSZ_integrand.shape}")
print(f"  DEBUG: kSZ_integrand RMS = {np.sqrt(np.mean(kSZ_integrand**2)):.4e}")
print("\n" + "="*70)
print("✓ CELL 5 COMPLETE")
print("="*70)


COMPUTING kSZ INTEGRAND WITH VISIBILITY FUNCTION (Per Seed)
Speed of light: c = 9.715612e-15 Mpc/s
✓ Computed kSZ integrand for 110 realisations

Generating kSZ Integrand Plot...
✓ Saved: kSZ_integrand_with_visibility_HII_EFF
  DEBUG: kSZ_integrand shape = (128, 128, 1753)
  DEBUG: kSZ_integrand RMS = 3.9284e-05

✓ CELL 5 COMPLETE


In [13]:
# %%
# =============================================================================
# CELL 6: Compute kSZ Map at Fixed z_obs = 5.0 (Per Seed)
# FIXED: Use ds_Mpc from tau_results (consistent with Cell 4)
# Changed z_obs to 5.0 to stay in reionization epoch
# =============================================================================



z_obs = 6.0   # Fixed observation redshift (in reionization epoch)

print("\n" + "="*70)
print(f"COMPUTING kSZ MAPS AT FIXED z_obs = {z_obs}")
print("="*70)

# Physical constants (CGS)
c_cm_s      = 3.0e10
sigma_T_cm2 = 6.6525e-25
n_e0_cm3    = 2.06e-7
Mpc_to_cm   = 3.0857e24
prefactor_cgs = n_e0_cm3 * sigma_T_cm2 * c_cm_s

print(f"Prefactor = {prefactor_cgs:.4e} s⁻¹")

# Storage: kSZ map at z_obs for each (seed, hii_eff)
kSZ_maps_at_zobs = {}

scan_start_time = time.time()

for (seed, hii_eff), lc in lightcones.items():
    if lc is None:
        continue
    
    if (seed, hii_eff) not in kSZ_integrand_results:
        continue
    
    if (seed, hii_eff) not in tau_results:
        continue
    
    print(f"  seed={seed:2d} | HII_EFF_FACTOR={hii_eff:.3f} ... ", end="")
    
    tau_data = tau_results[(seed, hii_eff)]
    kSZ_data = kSZ_integrand_results[(seed, hii_eff)]
    
    # Create cache directory
    kSZ_dir = os.path.join(main_cache_dir, f"seed_{seed:03d}_HIIEff{hii_eff:.4f}", "kSZ_maps")
    os.makedirs(kSZ_dir, exist_ok=True)
    
    # Extract data (use ds_Mpc from tau_results for consistency)
    red_axis      = np.asarray(tau_data['red_axis'], dtype=np.float64)
    z_mid         = np.asarray(tau_data['z_mid'], dtype=np.float64)
    ds_Mpc        = np.asarray(tau_data['ds_Mpc'], dtype=np.float64)
    kSZ_integrand = kSZ_data['kSZ_integrand']
    
    # Trim kSZ_integrand to match red_axis length
    if kSZ_integrand.shape[2] > len(red_axis):
        kSZ_integrand = kSZ_integrand[:, :, :len(red_axis)]
    
    ds_cm = ds_Mpc * Mpc_to_cm
    
    # Scale factor
    a = 1.0 / (1.0 + red_axis)
    a_squared_mid = 0.5 * (a[:-1]**2 + a[1:]**2)
    a_squared_mid_3D = a_squared_mid[None, None, :]
    kSZ_int_mid = 0.5 * (kSZ_integrand[:, :, :-1] + kSZ_integrand[:, :, 1:])
    
    kSZ_integrand_full = (prefactor_cgs / a_squared_mid_3D) * \
                         kSZ_int_mid * \
                         (ds_cm / c_cm_s)[None, None, :]
    
    # ===================================================================
    # Integrate from z_max down to z_obs = 8.0
    # ===================================================================
    idx_integrate = np.where(z_mid >= z_obs)[0]
    
    if len(idx_integrate) == 0:
        print(f"No slices beyond z={z_obs}")
        continue
    
    kSZ_map = np.sum(kSZ_integrand_full[:, :, idx_integrate], axis=2)
    
    # Save
    map_path = os.path.join(kSZ_dir, f"kSZ_map_z{z_obs:.1f}.npy")
    np.save(map_path, kSZ_map)
    
    # Store in memory
    kSZ_maps_at_zobs[(seed, hii_eff)] = {
        'kSZ_map': kSZ_map,
        'map_path': map_path,
        'z_obs': z_obs
    }
    
    rms = np.sqrt(np.mean(kSZ_map**2))
    print(f"Done | RMS = {rms:.4e}")

total_time = time.time() - scan_start_time
print(f"\n{'='*70}")
print(f"✓ kSZ MAPS at z={z_obs} COMPLETE")
print(f"Total time: {total_time/60:.2f} minutes")
print(f"Successful maps: {len(kSZ_maps_at_zobs)}")
print("="*70)


COMPUTING kSZ MAPS AT FIXED z_obs = 6.0
Prefactor = 4.1112e-21 s⁻¹
  seed= 1 | HII_EFF_FACTOR=52.000 ... Done | RMS = 1.6600e-07
  seed= 1 | HII_EFF_FACTOR=34.000 ... Done | RMS = 1.5259e-07
  seed= 1 | HII_EFF_FACTOR=38.500 ... Done | RMS = 1.5697e-07
  seed= 1 | HII_EFF_FACTOR=56.500 ... Done | RMS = 1.6866e-07
  seed= 1 | HII_EFF_FACTOR=43.000 ... Done | RMS = 1.6045e-07
  seed= 1 | HII_EFF_FACTOR=29.500 ... Done | RMS = 1.4704e-07
  seed= 1 | HII_EFF_FACTOR=47.500 ... Done | RMS = 2.0511e-07
  seed= 1 | HII_EFF_FACTOR=25.000 ... Done | RMS = 1.7834e-07
  seed= 2 | HII_EFF_FACTOR=29.500 ... Done | RMS = 1.3593e-07
  seed= 1 | HII_EFF_FACTOR=61.000 ... Done | RMS = 1.7114e-07
  seed= 1 | HII_EFF_FACTOR=65.500 ... Done | RMS = 1.7358e-07
  seed= 2 | HII_EFF_FACTOR=34.000 ... Done | RMS = 1.4017e-07
  seed= 1 | HII_EFF_FACTOR=70.000 ... Done | RMS = 2.2061e-07
  seed= 2 | HII_EFF_FACTOR=38.500 ... Done | RMS = 1.4325e-07
  seed= 2 | HII_EFF_FACTOR=25.000 ... Done | RMS = 1.6192e-07
  

In [14]:
# %%
# =============================================================================
# CELL 7: kSZ²-21cm Cross-Correlation Power Spectra at z_obs 
# Per (seed, hii_eff) — Proper error budget (sample + cosmic variance)
# =============================================================================

print("\n" + "="*70)
print("COMPUTING kSZ²-21cm CROSS-CORRELATION POWER SPECTRA (z_obs=8.0)")
print("="*70)

# =============================================================================
# Map & k-space Setup (Common to All)
# =============================================================================
npix_side    = user_params.HII_DIM
box_size_Mpc = float(user_params.BOX_LEN)
pix_size_Mpc = box_size_Mpc / npix_side
pix_area     = pix_size_Mpc**2

dk = 2 * np.pi / (npix_side * pix_size_Mpc)
kx = np.fft.fftshift(np.fft.fftfreq(npix_side)) * npix_side * dk
ky = np.fft.fftshift(np.fft.fftfreq(npix_side)) * npix_side * dk
kgrid = np.sqrt(kx[:, None]**2 + ky[None, :]**2)

k_bins    = np.logspace(np.log10(dk), np.log10(kgrid.max() * 0.9), 35)
k_centers = 0.5 * (k_bins[:-1] + k_bins[1:])

print(f"Map size: {npix_side}×{npix_side} pixels")
print(f"k-space: dk = {dk:.5f} Mpc⁻¹, {len(k_centers)} bins")

# =============================================================================
# Storage
# =============================================================================
cross_corr_all = {}   # key: (seed, hii_eff)

scan_start_time = time.time()

for (seed, hii_eff), lc in lightcones.items():
    if lc is None:
        continue
    
    if (seed, hii_eff) not in kSZ_maps_at_zobs:
        continue

    print(f"\n{'='*70}")
    print(f"Cross-correlation | seed={seed:2d} | HII_EFF_FACTOR={hii_eff:.3f}")
    print(f"{'='*70}")

    # Load kSZ map at z_obs
    kSZ_info = kSZ_maps_at_zobs[(seed, hii_eff)]
    kSZ_map = kSZ_info['kSZ_map']
    z_obs = kSZ_info['z_obs']

    # Square it
    kSZ2_map = kSZ_map**2
    kSZ2_centered = kSZ2_map - np.mean(kSZ2_map)
    fft_kSZ2_shifted = np.fft.fftshift(np.fft.fft2(kSZ2_centered))

    # Get 21cm slices up to z_obs
    lc_redshifts = np.asarray(lc.lightcone_redshifts, dtype=np.float64)
    valid_idx = np.where(lc_redshifts <= z_obs)[0]

    cross_corr_results = {}
    loop_start = time.time()

    for i, idx in enumerate(valid_idx):
        z_21cm = lc_redshifts[idx]

        T21_slice = np.asarray(lc.brightness_temp[:, :, idx])
        T21_centered = T21_slice - np.mean(T21_slice)
        fft_T21_shifted = np.fft.fftshift(np.fft.fft2(T21_centered))

        # Cross-power 2D
        cross_ps2d = np.real(np.conj(fft_kSZ2_shifted) * fft_T21_shifted) \
                     * pix_area / (npix_side**4)

        # Auto-powers
        auto_kSZ2_ps2d = np.abs(fft_kSZ2_shifted)**2 * pix_area / (npix_side**4)
        auto_T21_ps2d  = np.abs(fft_T21_shifted)**2  * pix_area / (npix_side**4)

        # ===================================================================
        # 1D Binning with Full Error Budget
        # ===================================================================
        C_cross_1d = np.zeros(len(k_centers))
        err_sample = np.zeros(len(k_centers))
        err_cosmic = np.zeros(len(k_centers))
        err_total  = np.zeros(len(k_centers))
        P_kSZ2_1d  = np.zeros(len(k_centers))
        P_T21_1d   = np.zeros(len(k_centers))

        for j in range(len(k_centers)):
            mask = (kgrid >= k_bins[j]) & (kgrid < k_bins[j+1])
            n_pix = np.sum(mask)

            if n_pix > 0:
                cross_vals = cross_ps2d[mask]
                
                C_cross_1d[j] = np.mean(cross_vals)
                
                # Sample variance
                err_sample[j] = np.std(cross_vals) / np.sqrt(n_pix)
                
                # Cosmic variance
                Pk1 = np.mean(auto_kSZ2_ps2d[mask])
                Pk2 = np.mean(auto_T21_ps2d[mask])
                err_cosmic[j] = np.sqrt(Pk1 * Pk2 + C_cross_1d[j]**2) / np.sqrt(Pk1 * Pk2) / np.sqrt(n_pix)
                
                # Total error
                err_total[j] = np.sqrt(err_sample[j]**2 + err_cosmic[j]**2)
                
                P_kSZ2_1d[j] = Pk1
                P_T21_1d[j]  = Pk2

        cross_corr_results[z_21cm] = {
            'k_centers': k_centers,
            'C_cross_1d': C_cross_1d,
            'C_cross_err_sample': err_sample,
            'C_cross_err_cosmic': err_cosmic,
            'C_cross_err_total':  err_total,
            'P_kSZ2_1d': P_kSZ2_1d,
            'P_T21_1d': P_T21_1d,
            'z_actual': float(z_21cm),
            'kSZ2_rms': float(np.sqrt(np.mean(kSZ2_map**2))),
            'T21_rms': float(np.sqrt(np.mean(T21_slice**2))),
            'T21_mean': float(np.mean(T21_slice))
        }

        if (i + 1) % 8 == 0 or i == 0 or i == len(valid_idx)-1:
            elapsed = time.time() - loop_start
            eta = (elapsed / (i+1)) * (len(valid_idx) - i - 1)
            sign = "+" if np.nanmean(C_cross_1d) > 0 else "-"
            print(f"  [{i+1:3d}/{len(valid_idx)}] z={z_21cm:.3f} | "
                  f"sign={sign} | ETA: {eta:.1f}s")

    loop_time = time.time() - loop_start

    cross_corr_all[(seed, hii_eff)] = {
        'cross_corr_results': cross_corr_results,
        'computation_time': loop_time,
        'z_obs': z_obs
    }

    print(f"✓ Completed seed={seed}, HIIEff={hii_eff:.3f} — {len(cross_corr_results)} redshifts")

total_time = time.time() - scan_start_time
print(f"\n{'='*70}")
print(f"ALL CROSS-CORRELATIONS COMPLETE — {total_time/60:.2f} minutes")
print(f"Processed {len(cross_corr_all)} realisations")
print("="*70)


COMPUTING kSZ²-21cm CROSS-CORRELATION POWER SPECTRA (z_obs=8.0)
Map size: 128×128 pixels
k-space: dk = 0.00785 Mpc⁻¹, 34 bins

Cross-correlation | seed= 1 | HII_EFF_FACTOR=52.000


/var/tmp/pbs.1549566.swarm/ipykernel_1250006/3930246430.py:105: RuntimeWarning: invalid value encountered in scalar divide
  err_cosmic[j] = np.sqrt(Pk1 * Pk2 + C_cross_1d[j]**2) / np.sqrt(Pk1 * Pk2) / np.sqrt(n_pix)


  [  1/77] z=5.000 | sign=- | ETA: 0.7s
  [  8/77] z=5.082 | sign=- | ETA: 0.3s
  [ 16/77] z=5.179 | sign=- | ETA: 0.2s
  [ 24/77] z=5.277 | sign=- | ETA: 0.2s
  [ 32/77] z=5.378 | sign=+ | ETA: 0.2s
  [ 40/77] z=5.481 | sign=- | ETA: 0.1s
  [ 48/77] z=5.587 | sign=- | ETA: 0.1s
  [ 56/77] z=5.696 | sign=+ | ETA: 0.1s
  [ 64/77] z=5.807 | sign=+ | ETA: 0.0s
  [ 72/77] z=5.921 | sign=- | ETA: 0.0s
  [ 77/77] z=5.993 | sign=+ | ETA: 0.0s
✓ Completed seed=1, HIIEff=52.000 — 77 redshifts

Cross-correlation | seed= 1 | HII_EFF_FACTOR=34.000
  [  1/77] z=5.000 | sign=- | ETA: 2.0s
  [  8/77] z=5.082 | sign=- | ETA: 0.4s
  [ 16/77] z=5.179 | sign=+ | ETA: 0.3s
  [ 24/77] z=5.277 | sign=- | ETA: 0.2s
  [ 32/77] z=5.378 | sign=+ | ETA: 0.2s
  [ 40/77] z=5.481 | sign=+ | ETA: 0.1s
  [ 48/77] z=5.587 | sign=- | ETA: 0.1s
  [ 56/77] z=5.696 | sign=+ | ETA: 0.1s
  [ 64/77] z=5.807 | sign=+ | ETA: 0.0s
  [ 72/77] z=5.921 | sign=+ | ETA: 0.0s
  [ 77/77] z=5.993 | sign=+ | ETA: 0.0s
✓ Completed seed=1

In [ ]:
# =============================================================================
# NOT FOR REPORT
# PLOT: kSZ, kSZ², and 21cm Maps Side-by-Side for Selected ION_Tvir_MIN
# Fixed x_e ~ 0.5, varying ION_Tvir_MIN
# =============================================================================

print(f"\n=== PLOTTING kSZ vs kSZ² vs 21cm MAPS FOR ION_Tvir_MIN SCAN ===")

# Target ionization fraction for comparison
target_xe = 0.5

# Select ION_Tvir_MIN values to plot
tvir_to_plot = tvir_subset  # Use the subset defined earlier

print(f"Plotting maps for {len(tvir_to_plot)} ION_Tvir_MIN values at x_e ~ {target_xe}")

# Create figure: rows = ION_Tvir_MIN, cols = [kSZ, kSZ², 21cm]
fig, axes = plt.subplots(len(tvir_to_plot), 3, 
                         figsize=(16, 5*len(tvir_to_plot)), 
                         constrained_layout=True)

if len(tvir_to_plot) == 1:
    axes = axes.reshape(1, -1)

for row_idx, tvir in enumerate(tvir_to_plot):
    
    if tvir not in lightcones or tvir not in cross_corr_all_tvir:
        # Fill with empty axes
        for col_idx in range(3):
            axes[row_idx, col_idx].text(0.5, 0.5, 'No data', 
                                        ha='center', va='center',
                                        transform=axes[row_idx, col_idx].transAxes,
                                        fontsize=16)
        continue
    
    lightcone = lightcones[tvir]
    kSZ_data = kSZ_maps_all_tvir[tvir]
    kSZ_maps_dir = kSZ_data['kSZ_maps_dir']
    
    # Find redshift closest to target x_e for this ION_Tvir_MIN
    z_nodes_sorted = np.asarray(lightcone.node_redshifts[::-1])
    x_e_nodes = 1.0 - lightcone.global_xH[::-1]
    
    idx_xe = np.argmin(np.abs(x_e_nodes - target_xe))
    z_obs = z_nodes_sorted[idx_xe]
    x_e = x_e_nodes[idx_xe]
    
    # Load kSZ map
    kSZ_map_file = f"{kSZ_maps_dir}/kSZ_map_z{z_obs:.6f}.npy"
    
    if not os.path.exists(kSZ_map_file):
        for col_idx in range(3):
            axes[row_idx, col_idx].text(0.5, 0.5, f'Map not found\nz={z_obs:.2f}', 
                                        ha='center', va='center',
                                        transform=axes[row_idx, col_idx].transAxes,
                                        fontsize=14)
        continue
    
    kSZ_map = np.load(kSZ_map_file)
    
    # Square it
    kSZ2_map = kSZ_map**2
    
    # Get lightcone redshift axis
    lc_redshifts = np.asarray(lightcone.lightcone_redshifts, dtype=np.float64)
    
    # Find closest lightcone slice to z_obs
    idx_closest = np.argmin(np.abs(lc_redshifts - z_obs))
    z_actual = lc_redshifts[idx_closest]
    
    # Extract 21cm brightness temperature slice
    T21_slice = np.asarray(lightcone.brightness_temp[:, :, idx_closest])
    
    # =============================================================================
    # Left panel: kSZ map
    # =============================================================================
    
    ax_kSZ = axes[row_idx, 0]
    
    # Symmetric color scale for kSZ
    vmax_kSZ = np.percentile(np.abs(kSZ_map), 99)
    
    im_kSZ = ax_kSZ.imshow(kSZ_map.T,
                           cmap='seismic',
                           origin='lower',
                           extent=[0, box_size_Mpc, 0, box_size_Mpc],
                           aspect='equal',
                           vmin=-vmax_kSZ,
                           vmax=vmax_kSZ)
    
    # Colorbar
    cbar_kSZ = plt.colorbar(im_kSZ, ax=ax_kSZ, fraction=0.046, pad=0.04)
    cbar_kSZ.set_label('kSZ', fontsize=11)
    cbar_kSZ.ax.tick_params(labelsize=9)
    
    # Labels
    ax_kSZ.set_xlabel('x [Mpc]', fontsize=12)
    ax_kSZ.set_ylabel('y [Mpc]', fontsize=12)
    ax_kSZ.set_title(f'kSZ Map\nTvir={tvir:.2f} (T={10**tvir:.1e}K)', 
                     fontsize=12, fontweight='bold')
    
    # Stats
    rms_kSZ = np.sqrt(np.mean(kSZ_map**2))
    ax_kSZ.text(0.05, 0.95, 
               f'z={z_obs:.2f}, $x_e$={x_e:.2f}\nRMS={rms_kSZ:.2e}',
               transform=ax_kSZ.transAxes, fontsize=10,
               verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # =============================================================================
    # Middle panel: kSZ² map
    # =============================================================================
    
    ax_kSZ2 = axes[row_idx, 1]
    
    # Use 'hot' colormap for squared map (all positive)
    vmax_kSZ2 = np.percentile(kSZ2_map, 99)
    
    im_kSZ2 = ax_kSZ2.imshow(kSZ2_map.T,
                             cmap='hot',
                             origin='lower',
                             extent=[0, box_size_Mpc, 0, box_size_Mpc],
                             aspect='equal',
                             vmin=0,
                             vmax=vmax_kSZ2)
    
    # Colorbar
    cbar_kSZ2 = plt.colorbar(im_kSZ2, ax=ax_kSZ2, fraction=0.046, pad=0.04)
    cbar_kSZ2.set_label('kSZ²', fontsize=11)
    cbar_kSZ2.ax.tick_params(labelsize=9)
    
    # Labels
    ax_kSZ2.set_xlabel('x [Mpc]', fontsize=12)
    ax_kSZ2.set_ylabel('y [Mpc]', fontsize=12)
    ax_kSZ2.set_title(f'kSZ² Map\nTvir={tvir:.2f} (T={10**tvir:.1e}K)', 
                      fontsize=12, fontweight='bold')
    
    # Stats
    rms_kSZ2 = np.sqrt(np.mean(kSZ2_map**2))
    ax_kSZ2.text(0.05, 0.95, 
                f'z={z_obs:.2f}, $x_e$={x_e:.2f}\nRMS={rms_kSZ2:.2e}',
                transform=ax_kSZ2.transAxes, fontsize=10,
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # =============================================================================
    # Right panel: 21cm brightness temperature map
    # =============================================================================
    
    ax_T21 = axes[row_idx, 2]
    
    # Use 'RdBu_r' for 21cm
    vmax_T21 = np.percentile(np.abs(T21_slice), 99)
    
    im_T21 = ax_T21.imshow(T21_slice.T,
                           cmap='RdBu_r',
                           origin='lower',
                           extent=[0, box_size_Mpc, 0, box_size_Mpc],
                           aspect='equal',
                           vmin=-vmax_T21,
                           vmax=vmax_T21)
    
    # Colorbar
    cbar_T21 = plt.colorbar(im_T21, ax=ax_T21, fraction=0.046, pad=0.04)
    cbar_T21.set_label('21cm [mK]', fontsize=11)
    cbar_T21.ax.tick_params(labelsize=9)
    
    # Labels
    ax_T21.set_xlabel('x [Mpc]', fontsize=12)
    ax_T21.set_ylabel('y [Mpc]', fontsize=12)
    ax_T21.set_title(f'21cm Map\nTvir={tvir:.2f} (T={10**tvir:.1e}K)', 
                     fontsize=12, fontweight='bold')
    
    # Stats
    rms_T21 = np.sqrt(np.mean(T21_slice**2))
    ax_T21.text(0.05, 0.95, 
               f'z={z_actual:.2f}, $x_e$={x_e:.2f}\nRMS={rms_T21:.1f}mK',
               transform=ax_T21.transAxes, fontsize=10,
               verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Overall title
fig.suptitle(f'Maps at $x_e$ ~ {target_xe} for Different ION_Tvir_MIN (HII_EFF={HII_EFF_FACTOR_FIXED})', 
            fontsize=20, fontweight='bold', y=0.995)

# Save
plot_name = "kSZ_kSZ2_21cm_maps_ION_Tvir_comparison"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

print("\n✓ MAP COMPARISON PLOTTING COMPLETE!")


=== PLOTTING kSZ vs kSZ² vs 21cm MAPS FOR ION_Tvir_MIN SCAN ===
Plotting maps for 4 ION_Tvir_MIN values at x_e ~ 0.5
✓ Saved: kSZ_kSZ2_21cm_maps_ION_Tvir_comparison

✓ MAP COMPARISON PLOTTING COMPLETE!


In [ ]:
# =============================================================================
# NOT FOR REPORT
# DIAGNOSTIC PLOTS: 2D FFT, Power Spectra vs k - ION_Tvir_MIN Scan
# =============================================================================

print(f"\n{'='*70}")
print("GENERATING DIAGNOSTIC PLOTS FOR ION_Tvir_MIN SCAN")
print(f"{'='*70}")

# =============================================================================
# PLOT 1: 2D FFT Maps (k-space) for Selected ION_Tvir_MIN at x_e ~ 0.5
# =============================================================================

print(f"\n=== PLOTTING 2D FFT MAPS IN k-SPACE ===")

# Target ionization fraction for comparison
target_xe = 0.5

# Select ION_Tvir_MIN values to plot
tvir_to_plot = tvir_subset

print(f"Plotting 2D FFT for {len(tvir_to_plot)} ION_Tvir_MIN values at x_e ~ {target_xe}")

# Create figure: rows = ION_Tvir_MIN, cols = [kSZ² FFT, 21cm FFT]
fig, axes = plt.subplots(len(tvir_to_plot), 2, 
                         figsize=(14, 6*len(tvir_to_plot)), 
                         constrained_layout=True)

if len(tvir_to_plot) == 1:
    axes = axes.reshape(1, -1)

for row_idx, tvir in enumerate(tvir_to_plot):
    
    if tvir not in lightcones or tvir not in cross_corr_all_tvir:
        for col_idx in range(2):
            axes[row_idx, col_idx].text(0.5, 0.5, 'No data', 
                                        ha='center', va='center',
                                        transform=axes[row_idx, col_idx].transAxes,
                                        fontsize=16)
        continue
    
    lightcone = lightcones[tvir]
    kSZ_data = kSZ_maps_all_tvir[tvir]
    kSZ_maps_dir = kSZ_data['kSZ_maps_dir']
    
    # Find redshift closest to target x_e
    z_nodes_sorted = np.asarray(lightcone.node_redshifts[::-1])
    x_e_nodes = 1.0 - lightcone.global_xH[::-1]
    
    idx_xe = np.argmin(np.abs(x_e_nodes - target_xe))
    z_obs = z_nodes_sorted[idx_xe]
    x_e = x_e_nodes[idx_xe]
    
    # Load kSZ map
    kSZ_map_file = f"{kSZ_maps_dir}/kSZ_map_z{z_obs:.6f}.npy"
    
    if not os.path.exists(kSZ_map_file):
        for col_idx in range(2):
            axes[row_idx, col_idx].text(0.5, 0.5, 'Map not found', 
                                        ha='center', va='center',
                                        transform=axes[row_idx, col_idx].transAxes,
                                        fontsize=14)
        continue
    
    kSZ_map = np.load(kSZ_map_file)
    kSZ2_map = kSZ_map**2
    kSZ2_map_centered = kSZ2_map - np.mean(kSZ2_map)
    
    # Get 21cm slice
    lc_redshifts = np.asarray(lightcone.lightcone_redshifts, dtype=np.float64)
    idx_closest = np.argmin(np.abs(lc_redshifts - z_obs))
    T21_slice = np.asarray(lightcone.brightness_temp[:, :, idx_closest])
    T21_slice_centered = T21_slice - np.mean(T21_slice)
    
    # Compute FFTs
    fft_kSZ2 = np.fft.fft2(kSZ2_map_centered)
    fft_kSZ2_shifted = np.fft.fftshift(fft_kSZ2)
    
    fft_T21 = np.fft.fft2(T21_slice_centered)
    fft_T21_shifted = np.fft.fftshift(fft_T21)
    
    # k-space extent
    k_max = kgrid.max()
    
    # =============================================================================
    # Left panel: kSZ² FFT
    # =============================================================================
    
    ax_fft_kSZ2 = axes[row_idx, 0]
    
    power_kSZ2 = np.abs(fft_kSZ2_shifted)**2
    power_kSZ2_log = np.log10(power_kSZ2 + 1e-20)
    
    im_fft_kSZ2 = ax_fft_kSZ2.imshow(power_kSZ2_log.T,
                                      cmap='viridis',
                                      origin='lower',
                                      extent=[-k_max, k_max, -k_max, k_max],
                                      aspect='equal')
    
    cbar_fft_kSZ2 = plt.colorbar(im_fft_kSZ2, ax=ax_fft_kSZ2, fraction=0.046, pad=0.04)
    cbar_fft_kSZ2.set_label(r'log$_{10}$(Power)', fontsize=11)
    cbar_fft_kSZ2.ax.tick_params(labelsize=9)
    
    ax_fft_kSZ2.set_xlabel(r'$k_x$ [Mpc$^{-1}$]', fontsize=12)
    ax_fft_kSZ2.set_ylabel(r'$k_y$ [Mpc$^{-1}$]', fontsize=12)
    ax_fft_kSZ2.set_title(f'kSZ² Power (k-space)\nTvir={tvir:.2f}, z={z_obs:.2f}, $x_e$={x_e:.2f}', 
                          fontsize=12, fontweight='bold')
    
    # Reference circle
    circle = plt.Circle((0, 0), 0.1, color='white', fill=False, linestyle='--', linewidth=1.5)
    ax_fft_kSZ2.add_patch(circle)
    
    # =============================================================================
    # Right panel: 21cm FFT
    # =============================================================================
    
    ax_fft_T21 = axes[row_idx, 1]
    
    power_T21 = np.abs(fft_T21_shifted)**2
    power_T21_log = np.log10(power_T21 + 1e-20)
    
    im_fft_T21 = ax_fft_T21.imshow(power_T21_log.T,
                                    cmap='viridis',
                                    origin='lower',
                                    extent=[-k_max, k_max, -k_max, k_max],
                                    aspect='equal')
    
    cbar_fft_T21 = plt.colorbar(im_fft_T21, ax=ax_fft_T21, fraction=0.046, pad=0.04)
    cbar_fft_T21.set_label(r'log$_{10}$(Power)', fontsize=11)
    cbar_fft_T21.ax.tick_params(labelsize=9)
    
    ax_fft_T21.set_xlabel(r'$k_x$ [Mpc$^{-1}$]', fontsize=12)
    ax_fft_T21.set_ylabel(r'$k_y$ [Mpc$^{-1}$]', fontsize=12)
    ax_fft_T21.set_title(f'21cm Power (k-space)\nTvir={tvir:.2f}, z={z_obs:.2f}, $x_e$={x_e:.2f}', 
                         fontsize=12, fontweight='bold')
    
    # Reference circle
    circle = plt.Circle((0, 0), 0.1, color='white', fill=False, linestyle='--', linewidth=1.5)
    ax_fft_T21.add_patch(circle)

fig.suptitle(f'2D Power Spectra at $x_e$ ~ {target_xe} (HII_EFF={HII_EFF_FACTOR_FIXED})', 
            fontsize=20, fontweight='bold', y=0.995)

plot_name = "2D_FFT_kspace_kSZ2_21cm_ION_Tvir"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT 2: Cross-Power vs k for Selected ION_Tvir_MIN at x_e ~ 0.5
# =============================================================================

print(f"\n=== PLOTTING CROSS-POWER vs k ===")

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

# Create colormap for ION_Tvir_MIN
cmap = mpl.cm.plasma
norm = mpl.colors.Normalize(vmin=ION_Tvir_MIN_VALUES.min(), vmax=ION_Tvir_MIN_VALUES.max())

for tvir in ION_Tvir_MIN_VALUES:
    
    if tvir not in cross_corr_all_tvir:
        continue
    
    lightcone = lightcones[tvir]
    cross_corr_results = cross_corr_all_tvir[tvir]['cross_corr_results']
    
    # Find redshift closest to target x_e
    z_nodes_sorted = np.asarray(lightcone.node_redshifts[::-1])
    x_e_nodes = 1.0 - lightcone.global_xH[::-1]
    
    idx_xe = np.argmin(np.abs(x_e_nodes - target_xe))
    z_obs = z_nodes_sorted[idx_xe]
    
    if z_obs not in cross_corr_results:
        continue
    
    results = cross_corr_results[z_obs]
    k_centers = results['k_centers']
    C_cross = results['C_cross_1d']
    
    valid = ~np.isnan(C_cross) & np.isfinite(C_cross)
    
    if np.sum(valid) > 5:
        color = cmap(norm(tvir))
        ax.plot(k_centers[valid], C_cross[valid], 
               color=color, linewidth=2, alpha=0.7,
               marker='o', markersize=3)

ax.set_xlabel(r'$k$ [Mpc$^{-1}$]', fontsize=18)
ax.set_ylabel(r'Cross-Power [Mpc$^2$]', fontsize=18)
ax.set_xscale('log')
ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.grid(True, alpha=0.3)

# Add colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label(r'ION\_Tvir\_MIN [log$_{10}$K]', fontsize=16)

ax.set_title(f'kSZ²-21cm Cross-Power at $x_e$ ~ {target_xe} (HII_EFF={HII_EFF_FACTOR_FIXED})', 
            fontsize=18, fontweight='bold')

plot_name = "cross_power_vs_k_ION_Tvir_xe05"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT 3: Cross-Power vs k for Single ION_Tvir_MIN, Multiple Redshifts
# =============================================================================

print(f"\n=== PLOTTING CROSS-POWER EVOLUTION FOR SINGLE ION_Tvir_MIN ===")

# Pick middle ION_Tvir_MIN value
tvir_middle = sorted(cross_corr_all_tvir.keys())[len(cross_corr_all_tvir)//2]

if tvir_middle in cross_corr_all_tvir:
    
    cross_corr_results = cross_corr_all_tvir[tvir_middle]['cross_corr_results']
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)
    
    # Sample every 15th redshift
    z_sample = sorted(cross_corr_results.keys())[::15]
    
    cmap_z = mpl.cm.rainbow
    norm_z = mpl.colors.Normalize(vmin=min(z_sample), vmax=max(z_sample))
    
    for z_obs in z_sample:
        results = cross_corr_results[z_obs]
        k_centers = results['k_centers']
        C_cross = results['C_cross_1d']
        
        valid = ~np.isnan(C_cross) & np.isfinite(C_cross)
        
        if np.sum(valid) > 5:
            color = cmap_z(norm_z(z_obs))
            ax.plot(k_centers[valid], C_cross[valid], 
                   color=color, linewidth=2, alpha=0.7,
                   marker='o', markersize=3)
    
    ax.set_xlabel(r'$k$ [Mpc$^{-1}$]', fontsize=18)
    ax.set_ylabel(r'Cross-Power [Mpc$^2$]', fontsize=18)
    ax.set_xscale('log')
    ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.grid(True, alpha=0.3)
    
    # Add colorbar
    sm = mpl.cm.ScalarMappable(cmap=cmap_z, norm=norm_z)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label(r'Redshift $z$', fontsize=16)
    
    ax.set_title(f'Cross-Power Evolution (Tvir={tvir_middle:.2f}, HII_EFF={HII_EFF_FACTOR_FIXED})', 
                fontsize=18, fontweight='bold')
    
    plot_name = f"cross_power_vs_k_z_evolution_Tvir{tvir_middle:.2f}"
    fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
    fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
    
    print(f"✓ Saved: {plot_name}")
    plt.close(fig)

print("\n✓ ALL DIAGNOSTIC PLOTTING COMPLETE!")


GENERATING DIAGNOSTIC PLOTS FOR ION_Tvir_MIN SCAN

=== PLOTTING 2D FFT MAPS IN k-SPACE ===
Plotting 2D FFT for 4 ION_Tvir_MIN values at x_e ~ 0.5
✓ Saved: 2D_FFT_kspace_kSZ2_21cm_ION_Tvir

=== PLOTTING CROSS-POWER vs k ===
✓ Saved: cross_power_vs_k_ION_Tvir_xe05

=== PLOTTING CROSS-POWER EVOLUTION FOR SINGLE ION_Tvir_MIN ===
✓ Saved: cross_power_vs_k_z_evolution_Tvir3.84

✓ ALL DIAGNOSTIC PLOTTING COMPLETE!


In [18]:
# %%
# =============================================================================
# CELL 8: Final Visualization — D_ℓ(ℓ=3000) + Reionization History + Milestones
# Three-panel plot (Main Science Plot)
# FIXED: k→ℓ conversion identical to solo code (with 0.67 factors)
# Colorbar: z_50 values
# =============================================================================

print("\n" + "="*70)
print("FINAL VISUALIZATION: D_ℓ(ℓ=3000) + Reionization History + Milestones")
print("="*70)

plot_dir_final = os.path.join(plot_dir, "plot_final_cell")
os.makedirs(plot_dir_final, exist_ok=True)
plot_dir_save = plot_dir_final

from astropy.cosmology import FlatLambdaCDM
cosmo = FlatLambdaCDM(H0=67.77, Om0=0.3086)
T_CMB_0_uK = 2.725 * 1e6

# =============================================================================
# Convert cross-power to ℓ-space with full error propagation
# Identical to solo code: ell = k * chi / 0.67, C_ell = power * 0.67^2 / D_A^2
# =============================================================================
cross_corr_ell_all = {}

for (seed, hii_eff), data in cross_corr_all.items():
    cross_corr_results = data.get('cross_corr_results', {})
    ell_results = {}
    
    for z_obs, res in cross_corr_results.items():
        D_A_Mpc = float(cosmo.angular_diameter_distance(z_obs).value)
        chi_comoving_Mpc = float(cosmo.comoving_distance(z_obs).value)
        T_CMB_z_uK = T_CMB_0_uK
        
        k_centers = res['k_centers']
        ell_from_k = k_centers * chi_comoving_Mpc / 0.67
        
        C_cross_ell = res['C_cross_1d'] * 0.67**2 / D_A_Mpc**2
        err_total = res.get('C_cross_err_total', res.get('C_cross_err', np.zeros_like(C_cross_ell))) * 0.67**2 / D_A_Mpc**2
        
        D_cross_ell = ell_from_k * (ell_from_k + 1) * C_cross_ell / (2 * np.pi)
        D_cross_ell_err = ell_from_k * (ell_from_k + 1) * err_total / (2 * np.pi)
        
        D_cross_ell_uK2_mK = D_cross_ell * T_CMB_z_uK**2
        D_cross_ell_uK2_mK_err = D_cross_ell_err * T_CMB_z_uK**2
        
        ell_results[z_obs] = {
            'ell_from_k': ell_from_k,
            'D_cross_ell_uK2_mK': D_cross_ell_uK2_mK,
            'D_cross_ell_uK2_mK_err': D_cross_ell_uK2_mK_err,
            'z_actual': z_obs
        }
    
    cross_corr_ell_all[(seed, hii_eff)] = ell_results

print(f"✓ Converted to ℓ-space for {len(cross_corr_ell_all)} realisations")

# =============================================================================
# Aggregate over seeds per HII_EFF_FACTOR
# =============================================================================
from collections import defaultdict
agg_data = defaultdict(list)

for (seed, hii_eff), res_dict in cross_corr_ell_all.items():
    for z, vals in res_dict.items():
        agg_data[(hii_eff, z)].append({
            'D_ell': vals['D_cross_ell_uK2_mK'],
            'D_err': vals['D_cross_ell_uK2_mK_err']
        })

final_stats = {}
for (hii_eff, z), lst in agg_data.items():
    D_vals = np.array([d['D_ell'] for d in lst])
    err_vals = np.array([d['D_err'] for d in lst])
    
    D_mean = np.mean(D_vals)
    D_seed_std = np.std(D_vals, ddof=1) if len(D_vals) > 1 else 0.0
    D_meas_err = np.mean(err_vals)
    D_total_err = np.sqrt(D_seed_std**2 + D_meas_err**2)
    
    final_stats[(hii_eff, z)] = {
        'D_mean': D_mean,
        'D_total_err': D_total_err
    }

# =============================================================================
# THREE-PANEL FINAL PLOT (Consistent colormap across all panels, no error bars)
# =============================================================================
fig = plt.figure(figsize=(20, 7), constrained_layout=True)
gs = fig.add_gridspec(1, 3)

ax_dl   = fig.add_subplot(gs[0, 0])   # D_ℓ at ℓ=3000
ax_xe   = fig.add_subplot(gs[0, 1])   # Reionization histories
ax_mil  = fig.add_subplot(gs[0, 2])   # Milestones

cmap = mpl.cm.plasma
norm = mpl.colors.Normalize(vmin=min(HII_EFF_FACTOR_VALUES), vmax=max(HII_EFF_FACTOR_VALUES))

# LEFT: D_ℓ(ℓ=3000) vs z (no error bars, symlog scale)
ell_target = 3000
for hii_eff in sorted(HII_EFF_FACTOR_VALUES):
    z_plot, D_plot = [], []
    for (he, z), stats in final_stats.items():
        if he == hii_eff:
            z_plot.append(z)
            D_plot.append(stats['D_mean'])
    
    if len(z_plot) > 0:
        color = cmap(norm(hii_eff))
        ax_dl.plot(z_plot, D_plot, color=color, lw=2.2, 
                   alpha=0.9, marker='o', markersize=6,
                   label=f'HIIEff={hii_eff:.2f}')

ax_dl.set_xlabel(r'Redshift $z$')
ax_dl.set_ylabel(r'$D_\ell(\ell=3000)$ [$\mu\mathrm{K}^2 \cdot \mathrm{mK}$]')
ax_dl.axhline(0, color='black', ls='--', lw=1)
ax_dl.invert_xaxis()
ax_dl.set_xlim(3, 6)
ax_dl.set_yscale('symlog', linthresh=1e-8)
#ax_dl.legend(title='HII_EFF_FACTOR', fontsize=11)

# MIDDLE: Reionization History
for hii_eff in HII_EFF_FACTOR_VALUES:
    found = False
    for (s, he), lc in lightcones.items():
        if he == hii_eff and lc is not None:
            z_nodes = np.asarray(lc.node_redshifts[::-1])
            x_e = 1.0 - np.asarray(lc.global_xH[::-1])
            color = cmap(norm(hii_eff))
            ax_xe.plot(z_nodes, x_e, color=color, lw=2.0, alpha=0.85)
            found = True
            break
    if found:
        continue

ax_xe.set_xlabel(r'Redshift $z$')
ax_xe.set_ylabel(r'Ionization Fraction $x_e$')
ax_xe.set_ylim(-0.05, 1.05)
ax_xe.invert_xaxis()

# RIGHT: Milestones
milestones = {}
for hii_eff in HII_EFF_FACTOR_VALUES:
    found = False
    for (s, he), lc in lightcones.items():
        if he == hii_eff and lc is not None:
            z_nodes = np.asarray(lc.node_redshifts[::-1])
            xe_nodes = 1.0 - np.asarray(lc.global_xH[::-1])
            z10 = z_nodes[np.argmin(np.abs(xe_nodes - 0.1))]
            z50 = z_nodes[np.argmin(np.abs(xe_nodes - 0.5))]
            z90 = z_nodes[np.argmin(np.abs(xe_nodes - 0.9))]
            milestones[hii_eff] = {'z10': z10, 'z50': z50, 'z90': z90}
            found = True
            break
    if found:
        continue

hii_effs = sorted(milestones.keys())
colors_10 = [cmap(norm(h)) for h in hii_effs]
colors_50 = [cmap(norm(h)) for h in hii_effs]
colors_90 = [cmap(norm(h)) for h in hii_effs]

ax_mil.scatter([milestones[h]['z50'] for h in hii_effs], 
               [milestones[h]['z10'] for h in hii_effs],
               c=colors_10, s=100, marker='o', label=r'$z_{10}$', edgecolors='black', linewidth=1)
ax_mil.scatter([milestones[h]['z50'] for h in hii_effs], 
               [milestones[h]['z50'] for h in hii_effs],
               c=colors_50, s=100, marker='s', label=r'$z_{50}$', edgecolors='black', linewidth=1)
ax_mil.scatter([milestones[h]['z50'] for h in hii_effs], 
               [milestones[h]['z90'] for h in hii_effs],
               c=colors_90, s=100, marker='^', label=r'$z_{90}$', edgecolors='black', linewidth=1)

ax_mil.set_xlabel(r'Reionization Midpoint $z_{50}$')
ax_mil.set_ylabel('Milestone Redshift')
ax_mil.legend()

# Colorbar - labeled by z_50 values
z50_values = [milestones[h]['z50'] for h in hii_effs]
z50_norm = mpl.colors.Normalize(vmin=min(z50_values), vmax=max(z50_values))
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=z50_norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=[ax_dl, ax_xe, ax_mil], pad=0.02, aspect=40)
cbar.set_label(r'$z_{50}$ (Reionization Midpoint)')

fig.suptitle(f'kSZ²–21cm at $\\ell=3000$ — HII_EFF_FACTOR Scan', 
             fontsize=18, fontweight='bold')

plot_name = "final_Dl3000_xe_milestones_HIIEff"
fig.savefig(f"{plot_dir_save}/{plot_name}.pdf", bbox_inches='tight')
fig.savefig(f"{plot_dir_save}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved main result: {plot_name}")
plt.close(fig)

print("\n" + "="*70)
print("FINAL SCIENCE PLOT COMPLETE!")
print("="*70)


FINAL VISUALIZATION: D_ℓ(ℓ=3000) + Reionization History + Milestones
✓ Converted to ℓ-space for 110 realisations
✓ Saved main result: final_Dl3000_xe_milestones_HIIEff

FINAL SCIENCE PLOT COMPLETE!
